In [1]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils.text import extract_text
import re
import json
import pandas as pd

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()
FORMULATION_PATH = Path("Formulación")

Extraer texto de la página

In [22]:
def extract_file_text(file):
    """Extrae el texto de cada página de un archivo pdf de formulación"""
    if isinstance(file, str):
        file = Path(file)
    with pdfplumber.open(file) as pdf:
        formulation_exercises = []
        for page in pdf.pages[1:]:
            exercise = page.extract_text()
            
            formulation_exercises.append(exercise)
    return formulation_exercises

Extraer nombre del examen

In [8]:
def extract_exam_name(statement):
    """Dependiendo del año, la forma de extrar el nombre del examen cambia."""
    lines = statement.split("\n")

    # 2023 y 2024
    if lines[0] == "R E S O L U C I Ó N":
        exam_line = lines[-1]
        return exam_line
    
    # buscar posición de cadena "RESOLUCIÓN"
    patron = re.compile(r"R\s*E\s*S\s*O\s*L\s*U\s*C\s*I\s*[ÓO]\s*N", re.IGNORECASE)
    m = patron.search(statement)
    end_pos = m.start() if m else -1
    
    # extraer nombre examen
    statement = statement[: end_pos]
    lines = statement.strip().split("\n")
    exam_name = lines[-1].lower()
    exam_name = re.sub(r"ejercicio \w+", "", exam_name)

    return exam_name

In [ ]:
print(extract_file_text( Path("Formulación/2025 - Formulación.pdf") )[0])

Extraer enunciado del ejercicio

In [57]:
def extract_nomenclature_statement(statement):
    
    # examen previo a 2025
    m = re.search(r"Formule o nombre los (?:siguientes compuestos|compuestos siguientes):? ", statement)
    if m:
        start_pos = m.end()
        end_pos = statement.find(" d)")
        statement = statement[start_pos:end_pos]

    # examen 2025
    else:
        lines = statement.split("\n")
        for idx, line in enumerate(lines):
            if line.startswith("Nombre o formule los siguientes compuestos"):
                start_idx = idx
        lines = lines[start_idx:]
    
        # buscar línea con nombres de compuestos orgánicos
        for idx, line in enumerate(lines):
                    if "c) " in line:
                        end_idx = idx
                        break

        statement = lines[end_idx]
        start_pos = statement.find("c) ")
        statement = statement[start_pos:]

    statement = (statement
                     .replace(";", " ")
                     .strip()
        )
    
    # remove duplicated spaces
    statement = re.sub(r"\s+", " ", statement)

    return statement

Extraer ejercicios (nombre examen y enunciado) de un pdf

In [47]:
def process_nomenclature_file(file):
    try:
        if isinstance(file, str):
                file = Path(file)

        year = int(file.name[:4])

        exercises = extract_file_content(file)

        tuples_list = []
        for ex in exercises:
            exam_name = extract_exam_name(ex)
            tuples_list.append((exam_name, extract_nomenclature_statement(ex)))

            
        return {year : tuples_list}
    except Exception as e:
         print(f"{file.name} - {e}")

In [59]:
form_emestrada_dict = {}
for year in range(2010, 2026):
    file = Path(f"Formulación/{year} - Formulación.pdf")
    try:
        form_emestrada_dict.update(process_nomenclature_file(file))
    except Exception as e:
        print(f"Año {year}. Error: {e}")
        

In [60]:
form_emestrada_dict

{2010: [('química. 2010. junio. . opción a',
   'a) Nitrito de plata b) Hidróxido de magnesio c) 1,1 Dicloroetano'),
  ('química. 2010. junio. . opción b',
   'a) Peróxido de rubidio b) Hidrogenocarbonato de calcio c) Butanona'),
  ('química. 2010. reserva 1. . opción a',
   'a) Permanganato de bario b) Peróxido de potasio c) Triclorometano'),
  ('química. 2010. reserva 1. . opción b',
   'a) Hidrogenocarbonato de sodio b) Hidróxido de cobre(I) c) Pent-1-ino'),
  ('química. 2010. reserva 2. . opción a',
   'a) Pentafluoruro de antimonio b) Óxido de plomo(II) c) Hex-4-en-2-ol'),
  ('química. 2010. reserva 2. . opción b',
   'a) Sulfato de manganeso (II) b) Hidróxido de cesio c) Fenol'),
  ('química. 2010. reserva 3. . opción a',
   'a) Sulfito de potasio b) Hidróxido de estroncio c) Ciclobutano'),
  ('química. 2010. reserva 3. . opción b',
   'a) Bromuro de hidrógeno b) Carbonato de calcio c) Ácido butanodioico'),
  ('química. 2010. reserva 4. . opción a',
   'a) Dióxido de azufre b) Ác

In [61]:
form_emestrada_list = [{"year" : year, "exam_name" : exam_name, "statement" : statement}
                       for year in form_emestrada_dict.keys()
                       for exam_name, statement in form_emestrada_dict[year]
                       ]

In [62]:
form_emestrada = pd.DataFrame(form_emestrada_list)

In [69]:
form_emestrada.sample(10)

,year,exam_name,statement
173,2024,química - 2024 - reserva 3,a) Óxido de aluminio b) Ácido cloroso c) But-2...
104,2018,química - 2018 - septiembre,a) Óxido de hierro(III) b) Hipoclorito de sodi...
175,2024,química - 2024 - reserva 4,a) Fluoruro de plata b) Ácido nitroso c) p-met...
143,2022,química - 2022 - junio -,a) Nitrato de hierro(III) b) Hidróxido de esta...
110,2019,química - 2019 - reserva 2,a) Peróxido de sodio b) Sulfuro de cinc c) p-M...
129,2020,química - 2020 - septiembre -,a) Sulfuro de cobre(I) b) Dióxido de azufre c)...
101,2018,química - 2018 - reserva 3,a) Trióxido de dicobalto b) Sulfito de sodio c...
103,2018,química - 2018 - reserva 4,a) Sulfuro de cadmio b) Ácido carbónico c) Pro...
36,2013,química - 2013 - junio,a) Hidróxido de paladio(II) b) Ácido sulfúrico...
122,2020,química - 2020 - reserva 2 -,a) Hidrogenocarbonato de plata b) Hidróxido de...


In [64]:
def clean_exam_name(exam_name):
    exam_name = re.sub(r"\. ejercicio (?:A[1-2]|4A)", "", exam_name, flags=re.IGNORECASE)
    exam_name = re.sub(r"\. \. opción [ab]", "", exam_name)
    # remove duplicated spaces
    exam_name = re.sub(r"\s+", " ", exam_name)
    exam_name = exam_name.replace(".", " -")

    return exam_name.lower()

In [67]:
# form_emestrada.exam_name.apply(clean_exam_name).unique()

In [68]:
form_emestrada.exam_name = form_emestrada.exam_name.apply(clean_exam_name)

In [70]:
form_emestrada.to_csv("form_emestrada.csv", index=False, encoding="utf-8")